In [18]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math

In [35]:
# The aim of the assignment is to simulate the BB84 key distribution protocol.

# This notebook is for a simulation of the protocol without an attacker.

simulator = BasicSimulator()
n = 24

def Generate_bits(n):
  circuit = QuantumCircuit(1,1)
  circuit.h(0)
  circuit.measure(0,0)
  result = simulator.run(circuit, shots=n, memory=True).result()
  bits = ''.join(result.get_memory())
  print(bits)
  return bits

def BasisToQubits(uBits, uBasis):
  noOfBits = len(uBasis)

  circuit = QuantumCircuit(noOfBits)
  #using odd and even numbers generated by math rand we chose to use diagonal or standard
  for i in range(noOfBits):
    if uBits[i] == '1':
      circuit.x(0)
    if uBasis[i] == '1':
      circuit.h(0)

  return circuit

personA_bits = Generate_bits(n)
personA_basis = Generate_bits(n)
personB_basis = Generate_bits(n)
print('person A bits ' + personA_bits)
print('person A basis ' + personA_basis)
print('person B basis ' + personB_basis)

#changing Person A basis into qubits
personA_qubits = BasisToQubits(personA_bits, personA_basis)

101110110111100001100011
001011000111110011011010
001000100100100101100011
person A bits 101110110111100001100011
person A basis 001011000111110011011010
person B basis 001000100100100101100011


In [36]:
#Person A tells Person B their basis to find out the bits

def ReceiveQubits(receivedQubits, generatedBasis):
  noOfBits = len(generatedBasis)

  for i in range(noOfBits):
      if personB_basis[i] == '1':
          receivedQubits.h(i)

  receivedQubits.measure_all()

  result = simulator.run(receivedQubits, shots=1, memory=True).result()

  received_bits = result.get_memory()[0]

  return received_bits

personB_bits = ReceiveQubits(personA_qubits, personB_basis)

print('Person B bits ' + personB_bits)


Person B bits 000001000000001000000001


In [41]:
#filter the bits

def FilterBits(uBitsA, uBitsB):
  noOfBits = len(uBitsA)
  filteredSender = ''
  filteredReceiver = ''
  for i in range(noOfBits):
    if (uBitsA[i] == uBitsB[i]):
      filteredSender = filteredSender + uBitsA[i]
      filteredReceiver = filteredReceiver + uBitsB[i]
  return filteredSender,filteredReceiver

senderKey, receiverKey = FilterBits(personA_bits, personB_bits)
print(senderKey)
print(receiverKey)

000000001
000000001
